# Swin Transformer Baseline Experiments for ISIC 2019

This notebook contains **baseline Swin-B experiments**, not the complete multimodal ensemble and two-stage calibration pipeline described in the paper.

## Contents
1. Download and extract the ISIC 2019 training-image archive.
2. Train a Swin-B model at 384 × 384 resolution using a fixed train/validation/test split and the DRW + class-balanced focal-loss schedule.
3. Fine-tune/evaluate a Swin-B model at 768 × 768 resolution, export pooled features, and generate evaluation reports.

## Before running
- Use a Colab runtime with a suitable GPU for practical training time.
- The image archive is downloaded by the notebook. The split CSV files (`train_split.csv`, `val_split.csv`, and `test_split.csv`) must be obtained separately and placed in the configured `DATASET_CSV_DIR` directory. These split files are not included in this repository.
- Review the editable path variables in the code before running. The default paths are generic examples; they may need to be changed for your environment.
- The 768 × 768 fine-tuning stage expects the 384 × 384 checkpoint at the configured project path. If the checkpoint is absent, the current code reports a warning and may start without those weights; verify this behavior before interpreting the run as fine-tuning.
- CAM/Grad-CAM visualization is optional and disabled by default in this cleaned copy.

## Reproducibility and outputs
All saved cell outputs and execution counts have been cleared so the notebook opens without stale logs, plots, predictions, or error traces. Running the cells will regenerate runtime outputs. Preserve any final metrics needed for the paper in versioned result files, rather than relying on notebook cell output alone.

Do not upload raw image data, private split files, checkpoints, or prediction-level files unless their redistribution and privacy conditions have been checked. The dataset should be obtained from its authorized source.


## Cell 1 — Download and extract the ISIC 2019 image archive

This cell downloads the ISIC 2019 training-image ZIP from the configured source, extracts it under `/content`, and checks that the expected image directory exists. It does **not** create the train/validation/test split CSV files; those are required by the training cells and must be supplied separately.

Run this cell once per fresh runtime, or skip the download when the archive has already been extracted.


In [ ]:
# Download and extract the ISIC 2019 training-image archive.
import os, zipfile, requests
from tqdm.auto import tqdm

URL = "https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip"
ZIP_PATH = "/content/ISIC_2019_Training_Input.zip"
EXTRACT_DIR = "/content"  # The archive is expected to contain the ISIC_2019_Training_Input folder.

os.makedirs(EXTRACT_DIR, exist_ok=True)

# -------- Download (percentage progress) --------
if not os.path.exists(ZIP_PATH):
    print("⬇️ Downloading:", URL)
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        pbar = tqdm(total=total, unit="B", unit_scale=True, unit_divisor=1024, desc="Downloading")
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
        pbar.close()
else:
    print("⚠️ ZIP archive already exists:", ZIP_PATH)

# -------- Extraction (percentage progress) --------
target_path = os.path.join(EXTRACT_DIR, "ISIC_2019_Training_Input")
if not os.path.exists(target_path):
    print("📦 Extracting to →", EXTRACT_DIR)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        members = zf.infolist()
        pbar = tqdm(total=len(members), desc="Extracting", unit="file")
        for m in members:
            zf.extract(m, EXTRACT_DIR)
            pbar.update(1)
        pbar.close()
else:
    print("⚠️ Target directory already exists:", target_path)

# -------- Final status --------
print("\n✅ Dataset is ready:", target_path)
print("First few files:", sorted(os.listdir(target_path))[:5])


## Cell 2 — Train the Swin-B baseline at 384 × 384

This cell prepares the runtime and paths, loads the fixed split CSV files, defines image transformations and the dataset/data loaders, constructs the pretrained Swin-B model, and trains with a cross-entropy warm-up followed by class-balanced focal loss. It then calculates classification metrics and report files; optional CAM visualizations can be enabled explicitly.

This is a **single-image baseline**. It does not by itself implement age/anatomical-region metadata fusion, the paper's multimodal ensemble, or the proposed two-stage probability calibration.

Before running, confirm `DATASET_CSV_DIR`, `IMAGE_DIR`, and `PROJECT_ROOT`. The original notebook's recorded run was interrupted before completion, so its stored outputs should not be treated as a complete run record.


In [ ]:
# Swin-B baseline training at 384 x 384 with DRW + class-balanced focal loss.

print("Installing required packages...")
try:
    import timm, pytorch_grad_cam
except Exception:
    !pip -q install timm grad-cam --upgrade

print("✅ Package installation complete.")

import os, json, math, random, zipfile, contextlib, warnings
warnings.filterwarnings("ignore")

import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from torchmetrics.classification import MulticlassF1Score, MulticlassAccuracy
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, matthews_corrcoef)
from sklearn.preprocessing import label_binarize

import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("Google Drive is already mounted, or the notebook is running outside Colab.")

# ==============================================================================
# ENVIRONMENT & PATHS (ONLY THE ROOT DIRECTORY CHANGED)
# ==============================================================================
PROJECT_ROOT = os.environ.get("SWIN_ROOT", "/content/drive/MyDrive/skin_lesion_project/swin")
CKPT_DIR  = os.path.join(PROJECT_ROOT, "checkpoints")
LOG_DIR   = os.path.join(PROJECT_ROOT, "logs")
FIG_DIR   = os.path.join(PROJECT_ROOT, "figs")
REP_DIR   = os.path.join(PROJECT_ROOT, "reports")
CAM_DIR   = os.path.join(PROJECT_ROOT, "cams")
PRED_DIR  = os.path.join(PROJECT_ROOT, "preds")
for d in [CKPT_DIR, LOG_DIR, FIG_DIR, REP_DIR, CAM_DIR, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

# Set to True only after checking the optional Grad-CAM dependency/API.
GENERATE_CAMS = False

best_model_path = os.path.join(CKPT_DIR, "best_model_swin384.pth")
last_model_path = os.path.join(CKPT_DIR, "last_model_swin384.pth")

DATASET_CSV_DIR = os.environ.get("DATASET_CSV_DIR", "/content/drive/MyDrive/skin_lesion_project/splits")
IMAGE_DIR = "/content/ISIC_2019_Training_Input"

# ==============================================================================
# REPRODUCIBILITY
# ==============================================================================
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

# Save environment version information
with open(os.path.join(LOG_DIR, "env_versions.txt"), "w") as f:
    f.write(f"torch={torch.__version__}\n")
    f.write(f"timm={timm.__version__}\n")
    try:
        import torchvision
        f.write(f"torchvision={torchvision.__version__}\n")
    except: pass
    try:
        import pytorch_grad_cam
        f.write(f"pytorch_grad_cam={pytorch_grad_cam.__version__}\n")
    except: pass
    f.write(f"numpy={np.__version__}\n")
    f.write(f"pandas={pd.__version__}\n")

# ==============================================================================
# HYPERPARAMETERS & TRANSFORMS (FIXED AT 384×384)
# ==============================================================================
IMAGE_SIZE = (384, 384)
batch_size = 64
base_lr = 2.5e-5
dropout_rate = 0.15
num_epochs = 300
num_workers = 4  # A value between 0 and 4 is a safe starting range in Colab

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE} | Image size: {IMAGE_SIZE} | num_workers={num_workers}")

train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ==============================================================================
# DATA PREPARATION (FIXED CSV SPLITS)
# ==============================================================================
if not os.path.exists(IMAGE_DIR):
    print(f"Image directory '{IMAGE_DIR}' not found. Downloading...")
    IMAGES_URL = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip"
    ZIP_PATH = '/content/ISIC_2019_Training_Input.zip'
    !wget -q --show-progress -O {ZIP_PATH} {IMAGES_URL}
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    !rm {ZIP_PATH}
    print("✅ Image data is ready.")
else:
    print(f"✅ Images found at: {IMAGE_DIR}")

print("Reading the fixed split CSV files...")
train_df = pd.read_csv(os.path.join(DATASET_CSV_DIR, "train_split.csv"))
val_df   = pd.read_csv(os.path.join(DATASET_CSV_DIR, "val_split.csv"))
test_df  = pd.read_csv(os.path.join(DATASET_CSV_DIR, "test_split.csv"))

class_names = sorted(train_df['class_name'].unique())
num_classes = len(class_names)
class_map = {name:i for i,name in enumerate(class_names)}
for df in [train_df, val_df, test_df]:
    df['label'] = df['class_name'].map(class_map)

print(f"Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)} | Classes={num_classes}")

# ==============================================================================
# DATASET / DATALOADER
# ==============================================================================
class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'] + ".jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        label = int(row['label'])
        return image, torch.tensor(label, dtype=torch.long), row['image'], img_path

train_dataset = ISICDataset(train_df, IMAGE_DIR, train_transform)
val_dataset   = ISICDataset(val_df,   IMAGE_DIR, val_test_transform)
test_dataset  = ISICDataset(test_df,  IMAGE_DIR, val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=(DEVICE=='cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=(DEVICE=='cuda'))

# ==============================================================================
# MODEL
# ==============================================================================
model = timm.create_model('swin_base_patch4_window12_384_in22k',
                          pretrained=True, num_classes=num_classes, drop_rate=dropout_rate)
model = model.to(DEVICE)

# ==============================================================================
# DRW + CLASS-BALANCED FOCAL (γ) — retain the existing structure
# ==============================================================================
# Effective number-based class weights (Cui et al.)
beta = 0.999  # The 0.99–0.9999 range is common; the choice depends on dataset size
cls_counts = train_df['label'].value_counts().sort_index().values.astype(np.float64)
effective_num = 1.0 - np.power(beta, cls_counts)
cb_weights = (1.0 - beta) / np.clip(effective_num, 1e-8, None)
cb_weights = cb_weights / cb_weights.sum() * num_classes  # normalize
cb_weights_t = torch.tensor(cb_weights, dtype=torch.float32, device=DEVICE)

gamma = 2.0  # focal gamma

def focal_loss_with_logits(logits, targets, weights=None, gamma=2.0, reduction='mean'):
    ce = nn.functional.cross_entropy(logits, targets, weight=weights, reduction='none')
    with torch.no_grad():
        pt = torch.softmax(logits, dim=1)[torch.arange(len(targets)), targets]
    fl = (1 - pt) ** gamma * ce
    return fl.mean() if reduction=='mean' else fl.sum()

criterion_ce = nn.CrossEntropyLoss()
def criterion_cb_focal(logits, targets):
    return focal_loss_with_logits(logits, targets, weights=cb_weights_t, gamma=gamma, reduction='mean')

optimizer = AdamW(model.parameters(), lr=base_lr, weight_decay=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

f1_metric = MulticlassF1Score(num_classes=num_classes, average='macro').to(DEVICE)
accuracy_metric = MulticlassAccuracy(num_classes=num_classes).to(DEVICE)
best_f1 = 0.0

use_amp = (DEVICE == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# DRW schedule: the first 33% uses CE; the remaining 67% uses CB-Focal
warmup_epochs = max(1, num_epochs // 3)

# Save configuration
config = dict(
    image_size=IMAGE_SIZE, batch_size=batch_size, base_lr=base_lr, dropout=dropout_rate,
    epochs=num_epochs, optimizer="AdamW", scheduler="CosineAnnealingLR",
    loss=f"DRW: CE({warmup_epochs}e) -> CB-Focal(gamma={gamma}, beta={beta})",
    seed=SEED, model="swin_base_patch4_window12_384_in22k"
)
with open(os.path.join(LOG_DIR, "config.json"), "w") as f: json.dump(config, f, indent=2)

# ==============================================================================
# TRAINING / VALIDATION
# ==============================================================================
def validate_model(model, loader, criterion_eval):
    model.eval()
    val_loss = 0.0
    f1_metric.reset(); accuracy_metric.reset()
    with torch.no_grad():
        for images, labels, _, _ in loader:
            images = images.to(DEVICE); labels = labels.to(DEVICE)
            outputs = model(images)
            loss = criterion_eval(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(1)
            f1_metric.update(preds, labels)
            accuracy_metric.update(preds, labels)
    return val_loss/len(loader), f1_metric.compute().item(), accuracy_metric.compute().item()

def train_model():
    global best_f1
    print("\nTraining started...")
    train_log, val_log = [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")

        # Select the loss function
        use_ce = (epoch < warmup_epochs)
        for images, labels, _, _ in pbar:
            images = images.to(DEVICE); labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if use_amp else contextlib.nullcontext()
            with ctx:
                outputs = model(images)
                loss = criterion_ce(outputs, labels) if use_ce else criterion_cb_focal(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            running_loss += loss.item()
            pbar.set_postfix(loss=f"{running_loss/(pbar.n+1):.4f}", phase=("CE" if use_ce else "CB-Focal"))

        scheduler.step()
        val_loss, val_f1, val_acc = validate_model(model, val_loader, criterion_ce)
        train_log.append([epoch+1, running_loss/len(train_loader)])
        val_log.append([epoch+1, val_loss, val_f1, val_acc])

        # Best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"🚀 New best Macro-F1: {best_f1:.4f} — model saved.")

        print(f"Epoch {epoch+1}: TrainLoss={running_loss/len(train_loader):.4f} | "
              f"ValLoss={val_loss:.4f} | ValF1={val_f1:.4f} | ValAcc={val_acc:.4f}")

    # Save the final model
    torch.save(model.state_dict(), last_model_path)

    # Save logs
    pd.DataFrame(train_log, columns=["epoch","train_loss"]).to_csv(os.path.join(LOG_DIR,"train_log.csv"), index=False)
    pd.DataFrame(val_log, columns=["epoch","val_loss","val_macroF1","val_acc"]).to_csv(os.path.join(LOG_DIR,"val_log.csv"), index=False)

# ==============================================================================
# TESTING, REPORTING & VISUALIZATIONS
# ==============================================================================
def save_confusion_matrices(cm, labels):
    # Raw
    plt.figure(figsize=(12,9))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels)
    plt.ylabel('Ground truth'); plt.xlabel('Predicted'); plt.title('Confusion Matrix (Raw)')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "cm_raw.png"), dpi=200); plt.close()

    # Normalize by true labels
    cmn = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(12,9))
    sns.heatmap(cmn, annot=True, fmt='.2f', xticklabels=labels, yticklabels=labels)
    plt.ylabel('Ground truth'); plt.xlabel('Predicted'); plt.title('Confusion Matrix (Normalized)')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "cm_norm.png"), dpi=200); plt.close()

def plot_roc_curves(y_true_bin, probs, labels):
    plt.figure(figsize=(12,9))
    per_auc = []
    for i in range(len(labels)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr); per_auc.append(roc_auc)
        plt.plot(fpr, tpr, label=f'{labels[i]} (AUC: {roc_auc:.4f})')
    # Micro and macro averages
    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), probs.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)
    per_auc_arr = np.array(per_auc)
    auc_macro = per_auc_arr.mean()
    plt.plot([0,1],[0,1],'k--',label='Rastgele')
    plt.title('Multiclass ROC (OvR)'); plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.legend(loc='lower right'); plt.grid(True)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"roc_ovr.png"), dpi=200); plt.close()
    return per_auc, auc_micro, auc_macro

def test_and_report():
    print("\n[Test] Loading the best model...")
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    model.eval()

    all_preds, all_labels, all_probs, all_ids, all_paths = [], [], [], [], []
    with torch.no_grad():
        for images, labels, img_ids, img_paths in tqdm(test_loader, desc="[Test]"):
            images = images.to(DEVICE); labels = labels.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(1).cpu().numpy()
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
            all_ids.extend(list(img_ids)); all_paths.extend(list(img_paths))

    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # CR
    cr_text = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=False)
    print("\n--- Classification Report ---\n", cr_text)
    cr_df = pd.DataFrame(classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)).transpose()
    cr_df.to_csv(os.path.join(REP_DIR, "classification_report.csv"))

    # CM
    cm = confusion_matrix(all_labels, all_preds)
    save_confusion_matrices(cm, class_names)

    # MCC
    mcc = matthews_corrcoef(all_labels, all_preds)

    # ROC & AUC
    y_true_bin = label_binarize(all_labels, classes=list(range(num_classes)))
    per_auc, auc_micro, auc_macro = plot_roc_curves(y_true_bin, all_probs, class_names)
    pd.DataFrame({"class": class_names, "auc": per_auc}).to_csv(os.path.join(REP_DIR, "per_class_auc.csv"), index=False)

    # Export predictions
    prob_cols = [f"p_{c}" for c in class_names]
    pred_dump = pd.DataFrame({
        "image_id": all_ids,
        "true_label": [class_names[i] for i in all_labels],
        "pred_label": [class_names[i] for i in all_preds],
    })
    pred_dump[prob_cols] = all_probs
    pred_dump.to_csv(os.path.join(PRED_DIR, "test_predictions.csv"), index=False)

    # Summary JSON
    metrics_summary = {
        "macroF1": float(cr_df.loc["macro avg","f1-score"]),
        "accuracy": float(cr_df.loc["accuracy","precision"]),  # scikit-learn stores the accuracy value in the "precision" column of the "accuracy" row
        "MCC": float(mcc),
        "AUC_micro": float(auc_micro),
        "AUC_macro": float(auc_macro),
        "classes": class_names
    }
    with open(os.path.join(REP_DIR, "metrics_summary.json"), "w") as f:
        json.dump(metrics_summary, f, indent=2)

    return all_ids, all_labels, all_preds, all_probs, pred_dump

# ==============================================================================
# CAM (Grad-CAM-like) — target the patch embedding layer for Swin
# ==============================================================================
def generate_cams(sample_df, n_tp=3, n_fp=3):
    # Select true-positive (TP) and false-positive (FP) samples
    df = sample_df.copy()
    df['true_idx'] = df['true_label'].map({c:i for i,c in enumerate(class_names)})
    df['pred_idx'] = df['pred_label'].map({c:i for i,c in enumerate(class_names)})

    # Collect TP and FP samples for each class
    from pytorch_grad_cam import GradCAM, EigenCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image

    # Swin: patch-embedding convolutional layer (always present)
    target_layer = model.patch_embed.proj

    os.makedirs(CAM_DIR, exist_ok=True)
    for ci, cname in enumerate(class_names):
        df_c = df[df['true_idx'] == ci]
        # TP
        tp = df_c[(df_c['true_idx']==df_c['pred_idx'])].sample(min(n_tp, len(df_c[(df_c['true_idx']==df_c['pred_idx'])])), random_state=SEED)
        # FP (samples predicted as this class but actually belonging to another class)
        fp = df[(df['pred_idx']==ci) & (df['true_idx']!=ci)].sample(min(n_fp, len(df[(df['pred_idx']==ci) & (df['true_idx']!=ci)])), random_state=SEED)

        for subset, tag in [(tp, "TP"), (fp, "FP")]:
            if len(subset)==0: continue
            for _, row in subset.iterrows():
                # Load the image and create an unnormalized float image for the overlay
                img_path = os.path.join(IMAGE_DIR, row['image_id'] + ".jpg")
                pil = Image.open(img_path).convert('RGB').resize(IMAGE_SIZE, Image.BILINEAR)
                rgb = np.array(pil).astype(np.float32) / 255.0

                tensor = val_test_transform(pil).unsqueeze(0).to(DEVICE)
                # Compute CAM (EigenCAM is more stable; GradCAM can also be used)
                try:
                    cam_algo = GradCAM(model=model, target_layers=[target_layer])
                except Exception:
                    cam_algo = EigenCAM(model=model, target_layers=[target_layer])

                targets = [ClassifierOutputTarget(int(row['pred_idx']))]  # Heatmap target for the predicted class
                grayscale_cam = cam_algo(input_tensor=tensor, targets=targets)[0]  # HxW [0,1]
                cam_vis = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)

                out_name = f"{cname}_{tag}_{row['image_id']}.png"
                Image.fromarray(cam_vis).save(os.path.join(CAM_DIR, out_name))

# ==============================================================================
# RUN
# ==============================================================================
if __name__ == "__main__":
    # Training
    train_model()
    # Test and reports
    dump = test_and_report()
    # Generate CAMs
    if GENERATE_CAMS:
        generate_cams(dump, n_tp=3, n_fp=3)
    print("\n✅ Completed. Outputs:")
    print(" - Checkpoints:", CKPT_DIR)
    print(" - Logs:", LOG_DIR)
    print(" - Reports:", REP_DIR)
    print(" - Figures:", FIG_DIR)
    print(" - CAMs:", CAM_DIR)
    print(" - Predictions:", PRED_DIR)


## Cell 3 — Fine-tune/evaluate Swin-B at 768 × 768 and export features

This cell configures the 768 × 768 experiment, optionally loads the 384 × 384 checkpoint, trains with early stopping, exports features and split metadata, and writes test metrics and reports. It expects the same split CSV files as the 384 × 384 cell.

Before running, confirm `DATASET_CSV_DIR`, `IMAGE_DIR`, and `SWIN_ROOT`. Verify that the expected 384 × 384 checkpoint exists if the goal is to reproduce a fine-tuning run rather than training from scratch. Feature exports include a test-split feature file; do not use test data for model selection or hyperparameter tuning.

CAM generation is disabled by default because it is an optional explanation step and the original run ended with a missing Grad-CAM package error. Enable it only after verifying the dependency and CAM API in your runtime.


In [ ]:
# -*- coding: utf-8 -*-
# Swin-B fine-tuning at 768 x 768 with feature export and evaluation reports.
print("Checking required packages...")
try:
    import timm, pytorch_grad_cam
except Exception:
    !pip -q install timm grad-cam --upgrade
print("✅")

import os, json, math, random, zipfile, contextlib, warnings
warnings.filterwarnings("ignore")

import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from torchmetrics.classification import MulticlassF1Score, MulticlassAccuracy
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, matthews_corrcoef)
from sklearn.preprocessing import label_binarize

import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# ==============================================================================
# PATHS — SEPARATE DIRECTORY FOR 768
# ==============================================================================
ROOT = os.environ.get("SWIN_ROOT", "/content/drive/MyDrive/skin_lesion_project/swin")
FT_ROOT = os.path.join(ROOT, "ft768")

CKPT_DIR  = os.path.join(FT_ROOT, "checkpoints")
LOG_DIR   = os.path.join(FT_ROOT, "logs")
FIG_DIR   = os.path.join(FT_ROOT, "figs")
REP_DIR   = os.path.join(FT_ROOT, "reports")
CAM_DIR   = os.path.join(FT_ROOT, "cams")
PRED_DIR  = os.path.join(FT_ROOT, "preds")
FEAT_DIR  = os.path.join(FT_ROOT, "features")
for d in [CKPT_DIR, LOG_DIR, FIG_DIR, REP_DIR, CAM_DIR, PRED_DIR, FEAT_DIR]:
    os.makedirs(d, exist_ok=True)

# Set to True only after checking the optional Grad-CAM dependency/API.
GENERATE_CAMS = False

best_model_path = os.path.join(CKPT_DIR, "best_model_swin768.pth")
last_model_path = os.path.join(CKPT_DIR, "last_model_swin768.pth")

# Path to the best 384 checkpoint for continued fine-tuning (change if needed)
previous_best_384 = os.path.join(ROOT, "checkpoints", "best_model_swin384.pth")

DATASET_CSV_DIR = os.environ.get("DATASET_CSV_DIR", "/content/drive/MyDrive/skin_lesion_project/splits")
IMAGE_DIR = "/content/ISIC_2019_Training_Input"

# ==============================================================================
# REPRODUCIBILITY
# ==============================================================================
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

with open(os.path.join(LOG_DIR, "env_versions.txt"), "w") as f:
    f.write(f"torch={torch.__version__}\n")
    f.write(f"timm={timm.__version__}\n")
    try:
        import torchvision
        f.write(f"torchvision={torchvision.__version__}\n")
    except: pass

# ==============================================================================
# 768×768 HYPERPARAMETERS
# ==============================================================================
IMAGE_SIZE = (768, 768)
batch_size = 8          # A conservative starting value for 768; increase if VRAM allows
base_lr = 2.5e-5
dropout_rate = 0.15
num_epochs = 200
patience = 20
num_workers = 4

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE={DEVICE} | SIZE={IMAGE_SIZE} | batch={batch_size} | epochs={num_epochs} | patience={patience}")

train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ==============================================================================
# DATA
# ==============================================================================
if not os.path.exists(IMAGE_DIR):
    print(f"Image directory '{IMAGE_DIR}' not found. Downloading...")
    IMAGES_URL = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip"
    ZIP_PATH = '/content/ISIC_2019_Training_Input.zip'
    !wget -q --show-progress -O {ZIP_PATH} {IMAGES_URL}
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    !rm {ZIP_PATH}
    print("✅ Image data is ready.")
else:
    print(f"✅ Images found at: {IMAGE_DIR}")

train_df = pd.read_csv(os.path.join(DATASET_CSV_DIR, "train_split.csv"))
val_df   = pd.read_csv(os.path.join(DATASET_CSV_DIR, "val_split.csv"))
test_df  = pd.read_csv(os.path.join(DATASET_CSV_DIR, "test_split.csv"))

class_names = sorted(train_df['class_name'].unique())
num_classes = len(class_names)
class_map = {name:i for i,name in enumerate(class_names)}
for df in [train_df, val_df, test_df]:
    df['label'] = df['class_name'].map(class_map)

print(f"Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)} | Classes={num_classes}")

# ==============================================================================
# DATASET / DATALOADER
# ==============================================================================
class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image'] + ".jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        label = int(row['label'])
        return image, torch.tensor(label, dtype=torch.long), row['image'], img_path

train_dataset = ISICDataset(train_df, IMAGE_DIR, train_transform)
val_dataset   = ISICDataset(val_df,   IMAGE_DIR, val_test_transform)
test_dataset  = ISICDataset(test_df,  IMAGE_DIR, val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=(DEVICE=='cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=(DEVICE=='cuda'))

# ==============================================================================
# MODEL — The Swin-B 384 architecture accepts 768-pixel inputs (window=12; relative-position bias table is fixed)
# ==============================================================================
# NEW: set img_size=768
model = timm.create_model(
    'swin_base_patch4_window12_384_in22k',
    pretrained=False,
    num_classes=num_classes,
    drop_rate=dropout_rate,
    img_size=768         # <-- kritik: 768 ile kur
).to(DEVICE)

# Fine-tune from 384: load directly if the classifier dimensions match
if os.path.exists(previous_best_384):
    try:
        state = torch.load(previous_best_384, map_location=DEVICE)
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"[FT] Loaded the 384 checkpoint. missing:{len(missing)} | unexpected:{len(unexpected)}")
    except Exception as e:
        print("[FT] Could not load the 384 checkpoint:", e)
else:
    print("[FT] Warning: 384 checkpoint not found; 768 training will start from scratch.")

# ==============================================================================
# DRW + CLASS-BALANCED FOCAL
# ==============================================================================
beta = 0.999
cls_counts = train_df['label'].value_counts().sort_index().values.astype(np.float64)
effective_num = 1.0 - np.power(beta, cls_counts)
cb_weights = (1.0 - beta) / np.clip(effective_num, 1e-8, None)
cb_weights = cb_weights / cb_weights.sum() * num_classes
cb_weights_t = torch.tensor(cb_weights, dtype=torch.float32, device=DEVICE)

gamma = 2.0
def focal_loss_with_logits(logits, targets, weights=None, gamma=2.0, reduction='mean'):
    ce = nn.functional.cross_entropy(logits, targets, weight=weights, reduction='none')
    with torch.no_grad():
        pt = torch.softmax(logits, dim=1)[torch.arange(len(targets)), targets]
    fl = (1 - pt) ** gamma * ce
    return fl.mean() if reduction=='mean' else fl.sum()

criterion_ce = nn.CrossEntropyLoss()
def criterion_cb_focal(logits, targets):
    return focal_loss_with_logits(logits, targets, weights=cb_weights_t, gamma=gamma, reduction='mean')

optimizer = AdamW(model.parameters(), lr=base_lr, weight_decay=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

f1_metric = MulticlassF1Score(num_classes=num_classes, average='macro').to(DEVICE)
acc_metric = MulticlassAccuracy(num_classes=num_classes).to(DEVICE)
best_f1 = 0.0; epochs_no_improve = 0

use_amp = (DEVICE == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

warmup_epochs = max(1, num_epochs // 3)  # ilk %33 CE, sonra CB-Focal

# Save configuration
config = dict(
    image_size=IMAGE_SIZE, batch_size=batch_size, lr=base_lr, dropout=dropout_rate,
    epochs=num_epochs, patience=patience, optimizer="AdamW", scheduler="CosineAnnealingLR",
    loss=f"DRW: CE({warmup_epochs}e) -> CB-Focal(gamma={gamma}, beta={beta})",
    seed=SEED, model="swin_base_patch4_window12_384_in22k", ft_from_384=os.path.exists(previous_best_384)
)
json.dump(config, open(os.path.join(LOG_DIR, "config.json"), "w"), indent=2)

# ==============================================================================
# TRAINING / VALIDATION
# ==============================================================================
def validate_model(model, loader, criterion_eval):
    model.eval()
    val_loss = 0.0
    f1_metric.reset(); acc_metric.reset()
    with torch.no_grad():
        for images, labels, _, _ in loader:
            images = images.to(DEVICE); labels = labels.to(DEVICE)
            outputs = model(images)
            loss = criterion_eval(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(1)
            f1_metric.update(preds, labels)
            acc_metric.update(preds, labels)
    return val_loss/len(loader), f1_metric.compute().item(), acc_metric.compute().item()

def train_model():
    global best_f1, epochs_no_improve
    print("\n[FT768] Training started...")
    train_log, val_log = [], []
    for epoch in range(num_epochs):
        model.train(); running = 0.0
        pbar = tqdm(train_loader, desc=f"[FT768] Epoch {epoch+1}/{num_epochs}")

        use_ce = (epoch < warmup_epochs)
        for images, labels, _, _ in pbar:
            images = images.to(DEVICE); labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if use_amp else contextlib.nullcontext()
            with ctx:
                outputs = model(images)
                loss = criterion_ce(outputs, labels) if use_ce else criterion_cb_focal(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            running += loss.item()
            pbar.set_postfix(loss=f"{running/(pbar.n+1):.4f}", phase=("CE" if use_ce else "CB-Focal"))

        scheduler.step()
        v_loss, v_f1, v_acc = validate_model(model, val_loader, criterion_ce)
        train_log.append([epoch+1, running/len(train_loader)])
        val_log.append([epoch+1, v_loss, v_f1, v_acc])

        # early stopping & best
        if v_f1 > best_f1:
            best_f1 = v_f1; epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"🚀 New best Macro-F1: {best_f1:.4f} — model saved.")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"⏹️  Early stopping (patience={patience})")
                break

        print(f"Epoch {epoch+1}: TrainLoss={running/len(train_loader):.4f} | "
              f"ValLoss={v_loss:.4f} | ValF1={v_f1:.4f} | ValAcc={v_acc:.4f}")

    torch.save(model.state_dict(), last_model_path)
    pd.DataFrame(train_log, columns=["epoch","train_loss"]).to_csv(os.path.join(LOG_DIR,"train_log.csv"), index=False)
    pd.DataFrame(val_log, columns=["epoch","val_loss","val_macroF1","val_acc"]).to_csv(os.path.join(LOG_DIR,"val_log.csv"), index=False)

# ==============================================================================
# TESTING & REPORTING (CM, CR, MCC, ROC & AUC) + FEATURE EXPORT
# ==============================================================================
def save_confusion_matrices(cm, labels):
    plt.figure(figsize=(12,9))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels)
    plt.ylabel('Ground truth'); plt.xlabel('Predicted'); plt.title('Confusion Matrix (Raw)')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "cm_raw.png"), dpi=200); plt.close()

    cmn = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(12,9))
    sns.heatmap(cmn, annot=True, fmt='.2f', xticklabels=labels, yticklabels=labels)
    plt.ylabel('Ground truth'); plt.xlabel('Predicted'); plt.title('Confusion Matrix (Normalized)')
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "cm_norm.png"), dpi=200); plt.close()

def plot_roc_curves(y_true_bin, probs, labels):
    plt.figure(figsize=(12,9))
    per_auc = []
    for i in range(len(labels)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs[:, i])
        roc_auc = auc(fpr, tpr); per_auc.append(roc_auc)
        plt.plot(fpr, tpr, label=f'{labels[i]} (AUC: {roc_auc:.4f})')
    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), probs.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)
    auc_macro = float(np.mean(per_auc))
    plt.plot([0,1],[0,1],'k--',label='Rastgele')
    plt.title('Multiclass ROC (OvR)'); plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.legend(loc='lower right'); plt.grid(True)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"roc_ovr.png"), dpi=200); plt.close()
    return per_auc, auc_micro, auc_macro

@torch.no_grad()
def export_features(model, loader, split_name):
    model.eval()
    feats, labels, ids = [], [], []
    for images, lbls, img_ids, _ in tqdm(loader, desc=f"[FT768] Export {split_name}"):
        images = images.to(DEVICE)
        # timm: use forward_head(pre_logits=True) for pre-logits
        x = model.forward_features(images)
        prelogits = model.forward_head(x, pre_logits=True)  # [B, D]
        feats.append(prelogits.float().cpu().numpy())
        labels.extend(lbls.numpy()); ids.extend(list(img_ids))
    feats = np.concatenate(feats, axis=0)
    np.save(os.path.join(FEAT_DIR, f"{split_name}_features.npy"), feats)
    pd.DataFrame({"image": ids, "label": labels}).to_csv(os.path.join(FEAT_DIR, f"{split_name}_meta.csv"), index=False)
    print(f"✅ {split_name}: features={feats.shape} saved.")

def test_and_report():
    print("\n[Test] Loading the best model...")
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    model.eval()

    all_preds, all_labels, all_probs, all_ids = [], [], [], []
    with torch.no_grad():
        for images, labels, img_ids, _ in tqdm(test_loader, desc="[Test]"):
            images = images.to(DEVICE); labels = labels.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(1).cpu().numpy()
            all_probs.append(probs); all_preds.append(preds); all_labels.append(labels.cpu().numpy())
            all_ids.extend(list(img_ids))

    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # Classification report (save and print)
    cr_txt = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
    print("\n=== Classification Report (TEST) ===\n", cr_txt)
    cr_df = pd.DataFrame(classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)).transpose()
    cr_df.to_csv(os.path.join(REP_DIR, "classification_report_test.csv"))

    # CM
    cm = confusion_matrix(all_labels, all_preds)
    save_confusion_matrices(cm, class_names)

    # MCC
    mcc = matthews_corrcoef(all_labels, all_preds)

    # ROC & AUC
    y_true_bin = label_binarize(all_labels, classes=list(range(num_classes)))
    per_auc, auc_micro, auc_macro = plot_roc_curves(y_true_bin, all_probs, class_names)
    pd.DataFrame({"class": class_names, "auc": per_auc}).to_csv(os.path.join(REP_DIR, "per_class_auc_test.csv"), index=False)

    # Export predictions
    prob_cols = [f"p_{c}" for c in class_names]
    dump = pd.DataFrame({
        "image_id": all_ids,
        "true_label": [class_names[i] for i in all_labels],
        "pred_label": [class_names[i] for i in all_preds],
    })
    dump[prob_cols] = all_probs
    dump.to_csv(os.path.join(PRED_DIR, "test_predictions.csv"), index=False)

    # Summary JSON + brief console summary
    summary = {
        "macroF1": float(cr_df.loc["macro avg","f1-score"]),
        "accuracy": float(cr_df.loc["accuracy","precision"]),
        "MCC": float(mcc),
        "AUC_micro": float(auc_micro),
        "AUC_macro": float(auc_macro),
        "classes": class_names
    }
    json.dump(summary, open(os.path.join(REP_DIR, "metrics_summary_test.json"), "w"), indent=2)

    print("\n=== TEST SUMMARY ===")
    print(f"Macro-F1: {summary['macroF1']:.4f} | MCC: {summary['MCC']:.4f} | "
          f"AUC micro/macro: {summary['AUC_micro']:.4f}/{summary['AUC_macro']:.4f}")

    return dump

# ==============================================================================
# CAM (optional; same method, target patch_embed)
# ==============================================================================
def generate_cams(sample_df, n_tp=3, n_fp=3):
    from pytorch_grad_cam import GradCAM, EigenCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image

    target_layer = model.patch_embed.proj
    os.makedirs(CAM_DIR, exist_ok=True)

    df = sample_df.copy()
    df['true_idx'] = df['true_label'].map({c:i for i,c in enumerate(class_names)})
    df['pred_idx'] = df['pred_label'].map({c:i for i,c in enumerate(class_names)})

    for ci, cname in enumerate(class_names):
        df_c = df[df['true_idx'] == ci]
        tp = df_c[df_c['true_idx']==df_c['pred_idx']].sample(min(n_tp, len(df_c[df_c['true_idx']==df_c['pred_idx']])), random_state=SEED)
        fp = df[(df['pred_idx']==ci) & (df['true_idx']!=ci)].sample(min(n_fp, len(df[(df['pred_idx']==ci) & (df['true_idx']!=ci)])), random_state=SEED)

        for subset, tag in [(tp, "TP"), (fp, "FP")]:
            if len(subset)==0: continue
            for _, row in subset.iterrows():
                img_path = os.path.join(IMAGE_DIR, row['image_id'] + ".jpg")
                pil = Image.open(img_path).convert('RGB').resize(IMAGE_SIZE, Image.BILINEAR)
                rgb = np.array(pil).astype(np.float32) / 255.0
                tensor = val_test_transform(pil).unsqueeze(0).to(DEVICE)
                try:
                    cam_algo = GradCAM(model=model, target_layers=[target_layer])
                except Exception:
                    cam_algo = EigenCAM(model=model, target_layers=[target_layer])
                targets = [ClassifierOutputTarget(int(row['pred_idx']))]
                grayscale_cam = cam_algo(input_tensor=tensor, targets=targets)[0]
                cam_vis = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)
                Image.fromarray(cam_vis).save(os.path.join(CAM_DIR, f"{cname}_{tag}_{row['image_id']}.png"))

# ==============================================================================
# RUN
# ==============================================================================
if __name__ == "__main__":
    # Training
    train_model()

    # Feature export (train/val/test → *.npy + *_meta.csv)
    @torch.no_grad()
    def _feat_loader(df, tfm):
        ds = ISICDataset(df, IMAGE_DIR, tfm)
        return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=(DEVICE=='cuda'))
    export_features(model, _feat_loader(train_df, val_test_transform), "train")
    export_features(model, _feat_loader(val_df,   val_test_transform), "val")
    export_features(model, _feat_loader(test_df,  val_test_transform), "test")

    # Test and reports
    dump = test_and_report()

    # CAM (optional; enable/disable)
    if GENERATE_CAMS:
        generate_cams(dump, n_tp=3, n_fp=3)

    print("\n✅ FT768 completed. Outputs:")
    print(" - Checkpoints:", CKPT_DIR)
    print(" - Logs:", LOG_DIR)
    print(" - Reports:", REP_DIR)
    print(" - Figures:", FIG_DIR)
    print(" - CAMs:", CAM_DIR)
    print(" - Predictions:", PRED_DIR)
    print(" - Features:", FEAT_DIR)
